# PROCESSING RAW SENSOR DATA

**Requirements**
- raw-data: raw sensor data files, with one file per experiment

#### Loading of necessary packages

In [ ]:
import json
import math
import elabapi_python
import numpy             as np
import pandas            as pd
import plotly.graph_objs as go
from datetime            import datetime
from pathlib             import Path
from plotly.subplots     import make_subplots
from elabapi_python.rest import ApiException

**Preparation for Communication with ELAB**

In [ ]:
#? COMMUNICATION WITH ELABFTW
api_client = ...
experimentsApi = elabapi_python.ExperimentsApi(api_client)
ItemsApi       = elabapi_python.ItemsApi(api_client)
UploadsApi     = elabapi_python.UploadsApi(api_client)

# list of available experiments and items
experiments              = experimentsApi.read_experiments(limit= 5000)
available_experiment_ids = [exp.id for exp in experiments]
items                    = ItemsApi.read_items(limit= 5000)
available_item_ids       = [item.id for item in items]

#### Initialization of needed variables

In [ ]:
raw_folder       = Path.cwd().parent / "data"    / "01_Timeseries_Sensors" / "raw"
Processed_folder = Path.cwd().parent / "data"    / "01_Timeseries_Sensors" / "processed_1"

First, we check, if data is available, that is not yet saved into the raw-data folder.

In [ ]:
# Go through all rows of the mapping file and check if the date is in the raw-data folder
mapping                  = ... # List of all experiments
missing_exp_ids          = []
missing_exp_dateversions = []

for value in mapping['operating_point_experiment_dateversion'].values:
    if not (raw_folder / f"{value}.csv").exists():
        print(f"File {value}.csv not found in raw-data folder. Skipping file.")
        # Collect the experiment IDs that are missing
        missing_exp_ids.append(mapping.loc[mapping['operating_point_experiment_dateversion'] == value, 'experiment_elab_id'].values[0])
        missing_exp_dateversions.append(value)
        continue

# Drop missing IDs ('-') from the collection
filtered = [(exp_id, exp_dateversion)
    for exp_id, exp_dateversion in zip(missing_exp_ids, missing_exp_dateversions)
    if exp_id != '-']

# Unzip the filtered list of tuples back into two lists
missing_exp_ids, missing_exp_dateversions = zip(*filtered) if filtered else ([], [])
if not missing_exp_ids:
    print("No IDs for missing experiments available.")
else:
    print(f"Missing experiment IDs: {missing_exp_ids}")
    print(f"Missing experiment dateversions: {missing_exp_dateversions}")

# Load the experiment from elab for each missing ID
for exp_id, exp_dateversion in zip(missing_exp_ids, missing_exp_dateversions):
    try:
        experiment = experimentsApi.get_experiment(exp_id)
        print(f"Experiment ({exp_id}) found.")
        sensor_data = []
        attached_files    = UploadsApi.read_uploads(entity_type="experiments",id=exp_id)
        for uploaded_file in attached_files:
            print(f"File {uploaded_file.real_name} found in experiment {exp_id}.")
            if "sensor_data" in uploaded_file.real_name:
                sensor_data_id   = uploaded_file.id  
                response         = UploadsApi.read_upload(entity_type="experiments", id=exp_id, subid=sensor_data_id, format="binary", _preload_content=False)  # Pull the binary payload • format='binary' tells the server not to JSON‑encode the response  • _preload_content=False prevents the client from trying to parse it
                temp_file        = Path.cwd() / "temp_sensor_data.csv"
                temp_file.write_bytes(response.data)
                temp_sensor_data = pd.read_csv(temp_file, header=0, sep=",", encoding="utf-8")
                sensor_data.append(temp_sensor_data)

        if sensor_data:
            sensor_data = pd.concat(sensor_data, ignore_index=True)
            # Rename the Time column to "Time" if it exists
            if "Timestamp" in sensor_data.columns:
                sensor_data.rename(columns={"Timestamp": "Time"}, inplace=True)
                # Convert the time to a datetime object from %H:%m:%s.xxx to %H:%M:%S
                sensor_data["Time"] = pd.to_datetime(sensor_data["Time"], format="%H:%M:%S.%f")
                # Drop the informtion about the day, month and year
                sensor_data["Time"] = sensor_data["Time"].dt.time
                sensor_data["Time"] = pd.to_datetime(sensor_data["Time"], format="%H:%M:%S.%f").dt.round('1s')
                sensor_data["Time"] = sensor_data["Time"].dt.time
                # Drop duplicate times
                sensor_data = sensor_data.drop_duplicates(subset=["Time"])

            # Add an index-column to the dataframe, starting with 0 at the first row
            sensor_data = sensor_data.reset_index()
            sensor_data = sensor_data.rename(columns={"index": "Index"})
            sensor_data = sensor_data[["Index", "Time", "LS701", "LS702", "AV8", "AV709", "P301", "AV716",  "H701", "H702", "H704", "H706", "H708", "TV1", "P701", "P702", "T701", "T702", "T703", "T704", "T706", "T708","T709", "T711", "T712", "T705","FI703", "FI704",  "PD1", "PD702", "PIC23", "FIA702"]]
            sensor_data.to_csv(raw_folder / f"{exp_dateversion}.csv", index=False)
            print(f"Sensor data for experiment {exp_id} saved to raw folder.")

            # Delete the temporary file
            temp_file.unlink(missing_ok=True)

    except ApiException as e:
        print(f"Error retrieving experiment {exp_id}: {e}")


For each experiment, the data is cut into three sections:  
- Startup: Starts when the heating is first turned on
- Regular operation: starts when both reflux and distillate stream are established
- Shutdown: starts when all heating is shut down

The decision when the regular operation starts, can be made automatically, or by human inspection

In [ ]:
for file in raw_folder.glob("*.csv"):
    experiment_id = file.stem

    # Check if there are already processed files in the processed_folder and skip them
    if (Processed_folder / ("Startup_" + file.name)).exists():
        print("File already processed: ", file.name)
        continue

    experiment          = experimentsApi.get_experiment(experiment_id)
    status              = experiment.status_title 
    experiment_links    = experiment.items_links

    if status == "Fail" or status == "No Distillation":
        print(f"Experiment {experiment_id} is not successful. Status: {status}. Skipping file {file.name}.")
        continue

    for entry in experiment_links:
        # If entry is a dict, use it directly; if it's an object, use __dict__
        json_entry = entry.__dict__        
        if json_entry.get("_category_title") == "Operating Point":
            operatingpoint_id    = json_entry.get("_entityid")
            operatingpoint_title = json_entry.get("_title")

    if not operatingpoint_id:
        print(f"No Operating Point found for experiment {experiment_id}. Skipping file {file.name}.")
        continue
    else:
        operatingpoint_entry                = ItemsApi.get_item(operatingpoint_id)
        operatingpoint_metadata             = json.loads(operatingpoint_entry.metadata)
        operating_point_heating_power_H002  = operatingpoint_metadata['extra_fields']["Specified Heating Power H002"]['value']

    # Load the data 
    data = pd.read_csv(file, sep=",")

    # Convert the columns "H701", "H702", "H706", "H704", "H708" to Watt
    data["H701"] = (data["H701"]/100) * 350                           # Convert to Watt
    data["H702"] = (data["H702"]/100) * 150                           # Convert to Watt
    data["H706"] = (data["H706"]/100) * 150                           # Convert to Watt
    data["H704"] = (data["H704"]/100) * 150                           # Convert to Watt
    data["H708"] = (data["H708"]/100) * 150                           # Convert to Watt

    # Add the column "H002" to the data frame
    data["H002"] = operating_point_heating_power_H002

    #* Split the data into three separate data frames
    #? Find the first index where "H701" is greater than 0
    try:
        index_startup = data.loc[data["H701"] > 0].index[0]
    except IndexError:
        print(f"No end of startup phase found in {file.name}. Skipping file.")
        continue

    #? Find the last index where "H701" is greater than 0
    index_shutdown = data.loc[data["H701"] > 0].index[-1]

    #? Find the start of the distillation phase
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,subplot_titles=(f"Temperatures in Experiment {file.stem}",f"Fluxes in Experiment {file.stem}"))
    temp_vars = ["T703", "T709", "T711", "T712", "T705"]
    for var in temp_vars:
        fig.add_trace(go.Scatter(x=data.index, y=data[var],     mode="lines", name=var),                              row=1, col=1)
    fig.add_trace(    go.Scatter(x=data.index, y=data["FI704"], mode="lines", name="FI704",line=dict(color="orange")),row=2, col=1)
    fig.add_trace(    go.Scatter(x=data.index, y=data["FI703"], mode="lines", name="FI703",line=dict(color="red")),   row=2, col=1)
    tick_vals = np.arange(0, len(data), 200)
    for r in [1, 2]:
        fig.update_xaxes(tickmode="array", tickvals=tick_vals, ticktext=tick_vals, tickangle=45, row=r, col=1)
    fig.update_yaxes(title_text="Temperature",        row=1, col=1)
    fig.update_yaxes(title_text="Flux", range=[0, 2], row=2, col=1)
    fig.update_layout(height=800, width=1200,legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.1),margin=dict(t=100))
    fig.show()

    # Ask the user if the start of the distillation phase should be determined automatically
    auto = input("Do you want to determine the distillation phase beginning index automatically? (y/n): ")
    if auto == "y":
        # Find the first index where "FI703" is greater than 0 for more than 5 seconds in a row
        mask_reflux        = ((data["FI703"] > 0.01) & (data["FI704"] > 0.01))
        runs               = mask_reflux.ne(mask_reflux.shift()).cumsum()                        # Label each run of identical mask‐values
        valid              = mask_reflux.groupby(runs).filter(lambda grp: grp.all() and len(grp) > 50)
        index_distillation = valid.index[0]
    else:
        # Query the user to define the distillation phase beginning index
        index_distillation = int(input("Define the distillation phase beginning index: "))

    #? Save the sections of the data
    # Define the "Startup" and "Distillation" phases
    Startup   = data.loc[index_startup:index_distillation]
    Operation = data.loc[index_distillation:index_shutdown-2]
    Shutdown  = data.loc[index_shutdown-2:]

    # Save the data frames with different prefixes "Startup_", "Distillation_", "Shutdown_" in the same folder
    Startup.to_csv(  Processed_folder / ("Startup_"   + file.name), index=False)
    Operation.to_csv(Processed_folder / ("Operation_" + file.name), index=False)
    Shutdown.to_csv( Processed_folder / ("Shutdown_"  + file.name), index=False)

# PROCESSING STEP 2

Here, several processing steps are done.  
- Renaming of feature names according to labels given in P&I flowsheet
- Adding of data quality feature
- Interpolating data for missing seconds

In [ ]:
folder_processed_1 = Path.cwd().parent / "data" / "01_Sensors_Timeseries" / "processed_1"
folder_processed_2 = Path.cwd().parent / "data" / "01_Sensors_Timeseries" / "processed_2"

In [ ]:
for file in folder_processed_1.glob("*.csv"):
    experiment_id = file.stem

    # Load the data
    data = pd.read_csv(file)

    # Correct the sensor data names
    data = data.rename(columns={"FI703": "FT703", "FI704": "FT704", "PD1": "PDI701", "PD702": "PDI702", "PIC23": "PY23", "FIA702": "FYI702",})

    # Read out the first and last time of the data
    first_time = data["Time"].iloc[0]
    last_time  = data["Time"].iloc[-1]
    data       = data.set_index("Time")
    data.index = pd.to_datetime(data.index, format="%H:%M:%S")

    # Remove duplicate labels
    data = data[~data.index.duplicated(keep='first')]

    # Add a new column "Data quality" with the value "1"
    # The values are given as follows:
    # 0 = No data
    # 1 = Data quality is good, data is actually measured live at the sensor / read out for the actor options
    # 2 = Data is interpolated, data is not actually measured live at the sensor / read out for the actor options
    # 3 = Data is extrapolated for some reason
    data["Data_quality"] = 1

    orig_idx     = data.index.copy()                # 1) Remember the original index and quality values
    orig_quality = data['Data_quality'].copy()

    # Interpolate the data to have a time step of 1 second
    data = data.resample("1s").interpolate(limit_direction="forward")
    quality               = pd.Series(2, index=data.index)        # Set the data quality to 2 for the interpolated data
    quality.loc[orig_idx] = orig_quality                          # overwrite originals with their true quality
    data['Data_quality']  = quality 
    data.index            = data.index.time                       # Drop the YYYY-MM-DD from the index

    # Drop "Unnamed: 0"
    if "Unnamed: 0" in data.columns:
        data = data.drop(columns="Unnamed: 0")
    elif "Index" in data.columns:
        data = data.drop(columns="Index")

    # Floor the entries for AV8, AV709, AV716, P301, LS701, LS702
    data["AV8"]   = np.floor(data["AV8"])
    data["AV716"] = np.floor(data["AV716"])
    data["AV709"] = np.floor(data["AV709"])
    data["P301"]  = np.floor(data["P301"])
    data["LS701"] = np.floor(data["LS701"])
    data["LS702"] = np.floor(data["LS702"])

    # Assign the Name "Time" to the index
    data.index.name = "Time"

    # Save the file to the folder "00_Sensor/processed_2"
    data.to_csv(folder_processed_2 / file.name)
    print("File saved: ", file.name)

# PROCESSING STEP 3: CREATE EXTRA TIMESERIES DATA FOR ACTUATORS

Also, the data-quality column is dropped.

In [ ]:
folder_sensor_processed_startup     = Path.cwd().parent / "data" / "01_Timeseries_Sensors"   / "final" / "Startup"
folder_sensor_processed_shutdown    = Path.cwd().parent / "data" / "01_Timeseries_Sensors"   / "final" / "Shutdown" 
folder_sensor_processed_operation   = Path.cwd().parent / "data" / "01_Timeseries_Sensors"   / "final" / "Operation"
folder_actuator_processed_startup   = Path.cwd().parent / "data" / "02_Timeseries_Actuators" / "final" / "Startup"
folder_actuator_processed_shutdown  = Path.cwd().parent / "data" / "02_Timeseries_Actuators" / "final" / "Shutdown"
folder_actuator_processed_operation = Path.cwd().parent / "data" / "02_Timeseries_Actuators" / "final" / "Operation"
folder_sensor_uncertainty           = Path.cwd().parent / "data" / "03_Timeseries_Sensor_Uncertainties" / "final" 

Actuator_features     = ["H701", "H702", "H706", "H704", "H708", "H002", "AV8", "AV709", "P301", "AV716", "TV1", "P701", "P702"]
Data_quality_features = ["Data_quality"]

def normalize_time_format(t):
    # Drop any empty rows
    if pd.isna(t) or t.strip() == "":
        return None
    try:
        # Try parsing as just time first
        return datetime.strptime(t, "%H:%M:%S").strftime("%H:%M:%S")
    except ValueError:
        # Fallback to datetime format
        return datetime.strptime(t, "%Y-%m-%d %H:%M:%S").strftime("%H:%M:%S")

In [ ]:
# Loop through all sensor data files in the startup folder
for file in folder_sensor_processed_startup.glob("*.csv"):
    print(f"Processing {file.stem} in Startup folder...") 
    # # Skip the file if already in actuator processed startup folder
    # if file.name in {f.name for f in folder_actuator_processed_startup.glob("*.csv")}:
    #     print(f"Skipping {file.name}, already processed.")
    #     continue

    # If the file does not exist in the sensor uncertainty folder, skip it and print a warning
    if not (folder_sensor_uncertainty / "Startup" / file.name).exists():
        print(f"Warning: {file.name} not found in sensor uncertainty folder. Skipping.")
        continue

    # Load the sensor data
    sensor_data         = pd.read_csv(file)
    sensor_data["Time"] = sensor_data["Time"].apply(normalize_time_format)
    # Change the sensor_data["Data_quality"] entries 1 to 0 and 2 to 1
    sensor_data["Data_quality"] = sensor_data["Data_quality"].replace({1: 0, 2: 1})

    # Load the sensor uncertainty data
    sensor_uncertainty = pd.read_csv(folder_sensor_uncertainty / "Startup" / file.name)
    # Add the "Data quality" column to the sensor uncertainty data frame
    sensor_uncertainty["Data_interpolated"] = sensor_data["Data_quality"]
    # Save the modified sensor uncertainty data frame to the sensor uncertainty folder
    sensor_uncertainty.to_csv(folder_sensor_uncertainty / "Startup" / file.name, index=False)

    # Put the Actuator features into a new data frame
    actuator_data         = sensor_data[Actuator_features].copy()
    actuator_data["Time"] = sensor_data["Time"]  # Add the "Time" column to the actuator data
    actuator_data.set_index("Time", inplace=True)  # Set the "Time" column as the index

    # Drop the actuator and data quality features from the sensor data
    sensor_data = sensor_data.drop(columns=Actuator_features)
    sensor_data = sensor_data.drop(columns=Data_quality_features)  

    # Save the actuator data to the actuator processed startup folder
    actuator_data.to_csv(folder_actuator_processed_startup / file.name, index=True)

    sensor_data.to_csv(folder_sensor_processed_startup / file.name, index=False)
    print(f"Processed {file.name} in Startup folder.")

# Loop through all sensor data files in the operation folder
for file in folder_sensor_processed_operation.glob("*.csv"):
    # # Skip the file if already in actuator processed operation folder
    # if file.name in {f.name for f in folder_actuator_processed_operation.glob("*.csv")}:
    #     print(f"Skipping {file.name}, already processed in actuator operation folder.")
    #     continue

    # If the file does not exist in the sensor uncertainty folder, skip it and print a warning
    if not (folder_sensor_uncertainty / "Operation" / file.name).exists():
        print(f"Warning: {file.name} not found in sensor uncertainty folder. Skipping.")
        continue

    # Load the sensor data
    sensor_data         = pd.read_csv(file)
    sensor_data["Time"] = sensor_data["Time"].apply(normalize_time_format)
    # Change the sensor_data["Data_quality"] entries 1 to 0 and 2 to 1
    sensor_data["Data_quality"] = sensor_data["Data_quality"].replace({1: 0, 2: 1})

    # Load the sensor uncertainty data
    sensor_uncertainty = pd.read_csv(folder_sensor_uncertainty / "Operation" / file.name)
    # Add the "Data quality" column to the sensor uncertainty data frame
    sensor_uncertainty["Data_interpolated"] = sensor_data["Data_quality"]
    # Save the modified sensor uncertainty data frame to the sensor uncertainty folder
    sensor_uncertainty.to_csv(folder_sensor_uncertainty / "Operation" / file.name, index=False)
    # Put the Actuator features into a new data frame
    actuator_data = sensor_data[Actuator_features].copy()
    actuator_data["Time"] = sensor_data["Time"]  # Add the "Time" column to the actuator data
    actuator_data.set_index("Time", inplace=True)  # Set the "Time" column as the index

    # Save the actuator data to the actuator processed operation folder
    actuator_data.to_csv(folder_actuator_processed_operation / file.name, index=True)

    # Drop the actuator and data quality features from the sensor data
    sensor_data = sensor_data.drop(columns=Actuator_features)
    sensor_data = sensor_data.drop(columns=Data_quality_features)
    sensor_data.to_csv(folder_sensor_processed_operation / file.name, index=False)
    print(f"Processed {file.name} in operation folder.")

# Loop through all sensor data files in the shutdown folder
for file in folder_sensor_processed_shutdown.glob("*.csv"):
    # # Skip the file if already in actuator processed shutdown folder
    # if file.name in {f.name for f in folder_actuator_processed_shutdown.glob("*.csv")}:
    #     print(f"Skipping {file.name}, already processed in actuator shutdown folder.")
    #     continue

    # If the file does not exist in the sensor uncertainty folder, skip it and print a warning
    if not (folder_sensor_uncertainty / "Shutdown" / file.name).exists():
        print(f"Warning: {file.name} not found in sensor uncertainty folder. Skipping.")
        continue

    # Load the sensor data
    sensor_data         = pd.read_csv(file)
    sensor_data["Time"] = sensor_data["Time"].apply(normalize_time_format)
    # Change the sensor_data["Data_quality"] entries 1 to 0 and 2 to 1
    sensor_data["Data_quality"] = sensor_data["Data_quality"].replace({1: 0, 2: 1})

    # Load the sensor uncertainty data
    sensor_uncertainty = pd.read_csv(folder_sensor_uncertainty / "Shutdown" / file.name)
    # Add the "Data quality" column to the sensor uncertainty data frame
    sensor_uncertainty["Data_interpolated"] = sensor_data["Data_quality"]
    # Save the modified sensor uncertainty data frame to the sensor uncertainty folder
    sensor_uncertainty.to_csv(folder_sensor_uncertainty / "Shutdown" / file.name, index=False)
    # Put the Actuator features into a new data frame
    actuator_data = sensor_data[Actuator_features].copy()
    actuator_data["Time"] = sensor_data["Time"]  # Add the "Time" column to the actuator data
    actuator_data.set_index("Time", inplace=True)  # Set the "Time" column as the index

    # Save the actuator data to the actuator processed shutdown folder
    actuator_data.to_csv(folder_actuator_processed_shutdown / file.name, index=True)

    # Drop the actuator and data quality features from the sensor data
    sensor_data = sensor_data.drop(columns=Actuator_features)
    sensor_data = sensor_data.drop(columns=Data_quality_features)
    sensor_data.to_csv(folder_sensor_processed_shutdown / file.name, index=False)
    print(f"Processed {file.name} in Shutdown folder.")